In [20]:
%load_ext autoreload
%autoreload 2
from utils.plotting import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
dir_path_seeds = np.array([
    # MLP 512
    # [
    #     "20260316-003710_FSNet_seed0_e300_lr1e-04_n7000",
    #     "20260316-004236_FSNet_seed1_e300_lr1e-04_n7000",
    #     "20260316-004343_FSNet_seed2_e300_lr1e-04_n7000",
    #     "20260316-004521_FSNet_seed3_e300_lr1e-04_n7000",
    # ],
    # MLP 1024
    [
        "20260315-162610_FSNet_seed0_e300_lr1e-04_n7000",
        "20260315-170538_FSNet_seed1_e300_lr1e-04_n7000",
        "20260315-172047_FSNet_seed2_e300_lr1e-04_n7000",
        "20260315-173358_FSNet_seed3_e300_lr1e-04_n7000",
    ],
    # MLP 2048
    [
        "20260315-181820_FSNet_seed0_e300_lr1e-04_n7000",
        "20260315-182727_FSNet_seed1_e300_lr1e-04_n7000",
        "20260315-183708_FSNet_seed2_e300_lr1e-04_n7000",
        "20260315-184525_FSNet_seed3_e300_lr1e-04_n7000",
    ],
    # MoE 256
    [
        "20260315-161700_FSNet_seed0_e300_lr1e-04_n7000_moe4k2_temp1.0_noise0.05",
        "20260315-162948_FSNet_seed1_e300_lr1e-04_n7000_moe4k2_temp1.0_noise0.05",
        "20260315-164153_FSNet_seed2_e300_lr1e-04_n7000_moe4k2_temp1.0_noise0.05",
        "20260315-165523_FSNet_seed3_e300_lr1e-04_n7000_moe4k2_temp1.0_noise0.05",
    ],
    # MoE 512
    [
        "20260315-170839_FSNet_seed0_e300_lr1e-04_n7000_moe4k2_temp1.0_noise0.05",
        "20260315-172157_FSNet_seed1_e300_lr1e-04_n7000_moe4k2_temp1.0_noise0.05",
        "20260315-173548_FSNet_seed2_e300_lr1e-04_n7000_moe4k2_temp1.0_noise0.05",
        "20260315-175041_FSNet_seed3_e300_lr1e-04_n7000_moe4k2_temp1.0_noise0.05",
    ],
])

In [22]:
# getting results
import os
import pickle
import yaml

rel_path = "./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000"

# collecting data
obj_mean = np.zeros_like(dir_path_seeds, dtype=float)
obj_max = np.zeros_like(dir_path_seeds, dtype=float)
eq_violation_l1_mean = np.zeros_like(dir_path_seeds, dtype=float)
eq_violation_l1_max = np.zeros_like(dir_path_seeds, dtype=float)
ineq_violation_l1_mean = np.zeros_like(dir_path_seeds, dtype=float)
ineq_violation_l1_max = np.zeros_like(dir_path_seeds, dtype=float)


def load_yaml_summary(path):
    """Load YAML summary with fallback for legacy python-tagged YAML files."""
    with open(path, "r") as f:
        text = f.read()

    try:
        return yaml.safe_load(text) or {}
    except yaml.constructor.ConstructorError:
        # Legacy summaries may contain Python object tags (e.g., TorchVersion).
        # These files are locally generated experiment artifacts.
        return yaml.unsafe_load(text) or {}


def load_test_metrics(dir_path, batch_size):
    """Load aggregated test metrics from new summary layout with old-layout fallback."""
    summary_path = os.path.join(dir_path, "test_summary.yaml")
    if os.path.exists(summary_path):
        summary = load_yaml_summary(summary_path)
        test_block = summary.get("test", {})

        # YAML may load numeric keys as int, old dumps may keep strings.
        bs_key = int(batch_size)
        result = test_block.get(bs_key, test_block.get(str(bs_key), None))
        if result is None:
            raise KeyError(f"batch size {batch_size} not found in {summary_path}")
        if "error" in result:
            raise RuntimeError(f"batch size {batch_size} has error: {result['error']}")
        return result

    # Backward compatibility: old results.pkl layout
    old_path = os.path.join(dir_path, "results.pkl")
    with open(old_path, "rb") as f:
        results = pickle.load(f)
    return results["test_results"]["batch_size_comparison"][batch_size]["metrics"]


batch_size = 256
for i in range(dir_path_seeds.shape[0]):  # over ckpt
    for j in range(dir_path_seeds.shape[1]):  # over seeds
        dir_path = os.path.join(rel_path, dir_path_seeds[i, j])
        print(dir_path)
        results_ = load_test_metrics(dir_path, batch_size)

        # opt_gap in saved files is already in percent
        obj_mean[i, j] = results_["opt_gap_mean"]
        obj_max[i, j] = results_["opt_gap_max"]
        eq_violation_l1_mean[i, j] = results_["eq_violation_l1_mean"]
        eq_violation_l1_max[i, j] = results_["eq_violation_l1_max"]
        ineq_violation_l1_mean[i, j] = results_["ineq_violation_l1_mean"]
        ineq_violation_l1_max[i, j] = results_["ineq_violation_l1_max"]

# assuming you already have: dir_path_seeds, rel_path
num_baselines = dir_path_seeds.shape[0]
num_seeds = dir_path_seeds.shape[1]
print(num_baselines, num_seeds)

# read one file to know number of epochs (still from detailed results.pkl)
with open(os.path.join(rel_path, dir_path_seeds[0, 0], "results.pkl"), "rb") as f:
    results = pickle.load(f)
    num_epochs = len(results["val_history"])

# store [num_baselines, num_seeds, num_epochs]
obj_mean_epochs = np.zeros((num_baselines, num_seeds, num_epochs))

for i in range(num_baselines):  # over ckpt
    for j in range(num_seeds):  # over seed
        dir_path = os.path.join(rel_path, dir_path_seeds[i, j])
        with open(os.path.join(dir_path, "results.pkl"), "rb") as f:
            results = pickle.load(f)
        # Optional: populate if val_history items include opt_gap_mean
        for k, entry in enumerate(results.get("val_history", [])):
            if isinstance(entry, dict) and "opt_gap_mean" in entry:
                obj_mean_epochs[i, j, k] = entry["opt_gap_mean"]
            else:
                obj_mean_epochs[i, j, k] = np.nan

./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260315-162610_FSNet_seed0_e300_lr1e-04_n7000
./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260315-170538_FSNet_seed1_e300_lr1e-04_n7000
./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260315-172047_FSNet_seed2_e300_lr1e-04_n7000
./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260315-173358_FSNet_seed3_e300_lr1e-04_n7000
./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260315-181820_FSNet_seed0_e300_lr1e-04_n7000
./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260315-182727_FSNet_seed1_e300_lr1e-04_n7000
./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260315-183708_FSNet_seed2_e300_lr1e-04_n7000
./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260315-184525_FSNet_seed3_e300_lr1e-04_n7000
./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260315-161700_FSNet_seed0_e300_lr1e-04_n7000_moe4k2_tem

In [23]:
# Define metrics and headers
metrics = [
    ("Opt Gap Mean", obj_mean),
    ("Opt Gap Max", obj_max),
    ("Eq Vio Mean", eq_violation_l1_mean),
    ("Eq Vio Max", eq_violation_l1_max),
    ("Ineq Vio Mean", ineq_violation_l1_mean),
    ("Ineq Vio Max", ineq_violation_l1_max),
]

# Print Header
header = f"{'Method':<8} | " + " | ".join([f"{name:<18}" for name, _ in metrics])
print(header)
print("-" * len(header))

# Print Rows (Method)
for i in range(num_baselines):
    row_str = f"{i:<8} | "
    for _, data in metrics:
        # Compute mean and std over seeds (axis 1)
        mu = np.mean(data[i])
        sigma = np.std(data[i])
        # Format as scientific notation
        if data is obj_mean or data is obj_max:
            row_str += f"{mu:.2f} ± {sigma:.2f}".ljust(18) + " | "
        else:
            row_str += f"{mu:.2e} ± {sigma:.2e}".ljust(18) + " | "
    print(row_str)

Method   | Opt Gap Mean       | Opt Gap Max        | Eq Vio Mean        | Eq Vio Max         | Ineq Vio Mean      | Ineq Vio Max      
--------------------------------------------------------------------------------------------------------------------------------------
0        | -246.25 ± 684.62   | 28668.13 ± 46632.77 | 6.52e-05 ± 1.99e-05 | 1.31e-03 ± 2.57e-04 | 4.75e-07 ± 5.55e-08 | 3.59e-05 ± 1.41e-05 | 
1        | 388.55 ± 716.77    | 76135.95 ± 44061.11 | 6.07e-05 ± 2.28e-05 | 1.34e-03 ± 2.98e-04 | 5.18e-07 ± 1.10e-07 | 4.39e-05 ± 1.61e-05 | 
2        | 505.81 ± 1086.62   | 101529.55 ± 139224.31 | 7.06e-05 ± 2.33e-05 | 1.17e-03 ± 1.28e-04 | 4.37e-07 ± 7.73e-08 | 3.70e-05 ± 1.23e-05 | 
3        | -103.58 ± 462.36   | 16608.28 ± 17853.45 | 3.94e-05 ± 1.21e-05 | 1.53e-03 ± 1.56e-04 | 4.85e-07 ± 7.92e-08 | 5.45e-05 ± 2.13e-05 | 


In [24]:
(-0.01 + 2)/2

0.995

In [25]:
(-0.02 + 1.7)/2

0.84

In [26]:
(((-0.02 + 0.01)/(-0.01)) + (1.7-2)/2)/2

0.425